In [2]:
import sys

PROJECT_ROOT = "/home/sagemaker-user/subocol-ia"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import pandas as pd
import joblib

from src.config import (
    S3_BUCKET,
    RAW_DATA_KEY,
    BEDROCK_KB_ID
)

from src.data.s3_io import load_csv_from_s3
from src.data.preprocessing import build_claim_dataset
from src.data.split_data import split_claim_data

from src.baseline.features import (
    create_feature_transformers,
    fit_transform_train_features,
    transform_features
)

from src.baseline.train import train_logistic_regression

from src.baseline.evaluate import (
    get_confusion_matrix,
    evaluate_predictions
)

from src.rag.prepare_documents import (
    build_training_documents,
    build_claim_query
)

from src.rag.retrieve import retrieve_similar_claims
from src.rag.classify import classify_claim_with_rag

In [5]:
df = load_csv_from_s3(
    bucket=S3_BUCKET,
    key=RAW_DATA_KEY
)

print("Raw dataset shape:", df.shape)

Raw dataset shape: (7532, 13)


In [6]:
claims_processed = build_claim_dataset(df)

print("Claim dataset shape:", claims_processed.shape)

Claim dataset shape: (800, 15)


In [7]:
train_df, val_df, test_df = split_claim_data(
    claims_processed
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (560, 15)
Validation: (120, 15)
Test: (120, 15)


In [8]:
(
    narrative_vectorizer,
    parts_vectorizer,
    brand_encoder,
    numeric_scaler
) = create_feature_transformers()

In [9]:
X_train = fit_transform_train_features(
    train_df,
    narrative_vectorizer,
    parts_vectorizer,
    brand_encoder,
    numeric_scaler
)

In [10]:
X_val = transform_features(
    val_df,
    narrative_vectorizer,
    parts_vectorizer,
    brand_encoder,
    numeric_scaler
)
X_test = transform_features(
    test_df,
    narrative_vectorizer,
    parts_vectorizer,
    brand_encoder,
    numeric_scaler
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (560, 5324)
X_val: (120, 5324)
X_test: (120, 5324)


In [11]:
y_train = train_df["estado_aviso"]
y_val = val_df["estado_aviso"]
y_test = test_df["estado_aviso"]

In [12]:
baseline_model = train_logistic_regression(
    X_train,
    y_train,
    C=1.0,
    class_weight="balanced"
)

In [13]:
objetado_index = list(
    baseline_model.classes_
).index("OBJETADO")

val_probabilities = baseline_model.predict_proba(
    X_val
)

test_probabilities = baseline_model.predict_proba(
    X_test
)

obj_val_probabilities = val_probabilities[
    :, objetado_index
]

obj_test_probabilities = test_probabilities[
    :, objetado_index
]

In [14]:
FINAL_THRESHOLD = 0.30

y_val_pred = np.where(
    obj_val_probabilities >= FINAL_THRESHOLD,
    "OBJETADO",
    "ENTREGADO"
)

y_test_pred = np.where(
    obj_test_probabilities >= FINAL_THRESHOLD,
    "OBJETADO",
    "ENTREGADO"
)

In [15]:
val_cm = get_confusion_matrix(
    y_val,
    y_val_pred
)

val_metrics = evaluate_predictions(
    y_val,
    y_val_pred
)

print("Validation confusion matrix:")
print(val_cm)

print("\nValidation metrics:")
print(val_metrics)

Validation confusion matrix:
[[ 9 36]
 [ 4 71]]

Validation metrics:
{'precision': 0.6635514018691588, 'recall': 0.9466666666666667, 'f1': 0.7802197802197802, 'f2': 0.8722358722358723}


In [16]:
test_cm = get_confusion_matrix(
    y_test,
    y_test_pred
)

test_metrics = evaluate_predictions(
    y_test,
    y_test_pred
)

print("Test confusion matrix:")
print(test_cm)

print("\nTest metrics:")
print(test_metrics)

Test confusion matrix:
[[ 2 43]
 [ 4 71]]

Test metrics:
{'precision': 0.6228070175438597, 'recall': 0.9466666666666667, 'f1': 0.7513227513227513, 'f2': 0.857487922705314}


In [17]:
baseline_bundle = {
    "model": baseline_model,
    "narrative_vectorizer": narrative_vectorizer,
    "parts_vectorizer": parts_vectorizer,
    "brand_encoder": brand_encoder,
    "numeric_scaler": numeric_scaler,
    "threshold": FINAL_THRESHOLD
}

In [18]:
from pathlib import Path
import joblib

MODEL_PATH = Path(PROJECT_ROOT) / "models" / "baseline_bundle.joblib"

joblib.dump(
    baseline_bundle,
    MODEL_PATH
)

print("Saved baseline bundle to:")
print(MODEL_PATH)

print("\nFile exists:", MODEL_PATH.exists())

Saved baseline bundle to:
/home/sagemaker-user/subocol-ia/models/baseline_bundle.joblib

File exists: True


In [19]:
from src.data.s3_io import upload_file_to_s3
BASELINE_MODEL_KEY = "models/baseline_bundle.joblib"

upload_file_to_s3(
    local_path=MODEL_PATH,
    bucket=S3_BUCKET,
    key=BASELINE_MODEL_KEY
)

print(
    f"Uploaded to s3://{S3_BUCKET}/{BASELINE_MODEL_KEY}"
)

Uploaded to s3://subocol-ia-1234/models/baseline_bundle.joblib


In [20]:
training_documents = build_training_documents(
    train_df
)

print("Training documents:", len(training_documents))

Training documents: 560


In [21]:
sample_id = next(iter(training_documents))

print(training_documents[sample_id])

ID del aviso: 219266

Fecha de creación: 2024-11-14 07:40:21.755000

Vehículo:
Marca: MAZDA
Línea: CX5 [2]
Versión: TOURING TP 2000CC 6AB R17 4X2
Modelo: 2020
Edad del vehículo: 4

Versión de los hechos:
caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com - vehiculos afectados: no - otros afectados: caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com

Piezas inspeccionadas:
bocel ampliacion delantero izquierdo guardafango izquierdo

Número total de piezas: 2
Número de piezas con descripción válida: 2

Decisión históric

In [22]:
from src.data.s3_io import (
    load_csv_from_s3,
    upload_documents_to_s3,
    list_s3_objects
)

In [23]:
RAG_TRAIN_PREFIX = "rag/train"

upload_documents_to_s3(
    bucket=S3_BUCKET,
    prefix=RAG_TRAIN_PREFIX,
    documents=training_documents
)

In [24]:
rag_files = list_s3_objects(
    S3_BUCKET,
    f"{RAG_TRAIN_PREFIX}/"
)

print("RAG documents in S3:", len(rag_files))

RAG documents in S3: 560


 ## Stage 3 — Retrieval-Augmented Generation (RAG)

In [25]:
RAG_TOP_K = 5
sample_claim = val_df.iloc[0]

sample_query = build_claim_query(
    sample_claim
)

print(sample_query)

Vehículo:
Marca: TOYOTA
Línea: HILUX [8] [2 FL]
Versión: 2.8L TP 2800CC TD 7AB 4X4 EURO IV
Modelo: 2023
Edad del vehículo: 1

Versión de los hechos:
impacto en la parte trasera de mi vehiculo frene me di cuenta que me habia chocado una motocicleta

Piezas inspeccionadas:
bocel bomper trasero derecho puntera derecho bomper trasero sensores de aproximacion stop derecho refuerzo central bomper trasero

Número total de piezas: 5
Número de piezas con descripción válida: 5


In [26]:
sample_retrieved = retrieve_similar_claims(
    query_text=sample_query,
    knowledge_base_id=BEDROCK_KB_ID,
    number_of_results=RAG_TOP_K
)

print(
    "Retrieved historical claims:",
    len(sample_retrieved)
)

Retrieved historical claims: 5


In [27]:
sample_rag_prediction = classify_claim_with_rag(
    current_claim_text=sample_query,
    retrieved_claims=sample_retrieved
)

print(
    "RAG prediction:",
    sample_rag_prediction
)

print(
    "Actual label:",
    sample_claim["estado_aviso"]
)

RAG prediction: ENTREGADO
Actual label: OBJETADO


In [28]:
rag_val_results = []

for i, (_, claim) in enumerate(val_df.iterrows(), start=1):

    query = build_claim_query(claim)

    retrieved = retrieve_similar_claims(
        query_text=query,
        knowledge_base_id=BEDROCK_KB_ID,
        number_of_results=RAG_TOP_K
    )

    prediction = classify_claim_with_rag(
        current_claim_text=query,
        retrieved_claims=retrieved
    )

    rag_val_results.append({
        "numero_aviso": claim["numero_aviso"],
        "actual": claim["estado_aviso"],
        "prediction": prediction
    })

    print(f"Processed {i}/{len(val_df)}")

Processed 1/120
Processed 2/120
Processed 3/120
Processed 4/120
Processed 5/120
Processed 6/120
Processed 7/120
Processed 8/120
Processed 9/120
Processed 10/120
Processed 11/120
Processed 12/120
Processed 13/120
Processed 14/120
Processed 15/120
Processed 16/120
Processed 17/120
Processed 18/120
Processed 19/120
Processed 20/120
Processed 21/120
Processed 22/120
Processed 23/120
Processed 24/120
Processed 25/120
Processed 26/120
Processed 27/120
Processed 28/120
Processed 29/120
Processed 30/120
Processed 31/120
Processed 32/120
Processed 33/120
Processed 34/120
Processed 35/120
Processed 36/120
Processed 37/120
Processed 38/120
Processed 39/120
Processed 40/120
Processed 41/120
Processed 42/120
Processed 43/120
Processed 44/120
Processed 45/120
Processed 46/120
Processed 47/120
Processed 48/120
Processed 49/120
Processed 50/120
Processed 51/120
Processed 52/120
Processed 53/120
Processed 54/120
Processed 55/120
Processed 56/120
Processed 57/120
Processed 58/120
Processed 59/120
Proces

In [29]:
rag_val_df = pd.DataFrame(rag_val_results)

rag_val_df.head()

,numero_aviso,actual,prediction
0,189957,OBJETADO,ENTREGADO
1,131524,OBJETADO,OBJETADO
2,133876,OBJETADO,OBJETADO
3,195151,ENTREGADO,ENTREGADO
4,243983,OBJETADO,OBJETADO


In [30]:
rag_val_cm = get_confusion_matrix(
    rag_val_df["actual"],
    rag_val_df["prediction"]
)

rag_val_metrics = evaluate_predictions(
    rag_val_df["actual"],
    rag_val_df["prediction"]
)

tn, fp, fn, tp = rag_val_cm.ravel()

rag_val_specificity = tn / (tn + fp)

print("RAG validation confusion matrix:")
print(rag_val_cm)

print("\nRAG validation metrics:")
print(rag_val_metrics)

print("\nSpecificity:", rag_val_specificity)

RAG validation confusion matrix:
[[21 24]
 [15 60]]

RAG validation metrics:
{'precision': 0.7142857142857143, 'recall': 0.8, 'f1': 0.7547169811320755, 'f2': 0.78125}

Specificity: 0.4666666666666667


In [31]:
RAG_VAL_RESULTS_PATH = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "rag_validation_results.csv"
)

RAG_VAL_RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

rag_val_df.to_csv(
    RAG_VAL_RESULTS_PATH,
    index=False
)

print("Saved to:")
print(RAG_VAL_RESULTS_PATH)

Saved to:
/home/sagemaker-user/subocol-ia/data/processed/rag_validation_results.csv


In [32]:
rag_test_results = []

for i, (_, claim) in enumerate(test_df.iterrows(), start=1):

    query = build_claim_query(claim)

    retrieved = retrieve_similar_claims(
        query_text=query,
        knowledge_base_id=BEDROCK_KB_ID,
        number_of_results=RAG_TOP_K
    )

    prediction = classify_claim_with_rag(
        current_claim_text=query,
        retrieved_claims=retrieved
    )

    rag_test_results.append({
        "numero_aviso": claim["numero_aviso"],
        "actual": claim["estado_aviso"],
        "prediction": prediction
    })

    print(f"Processed {i}/{len(test_df)}")

Processed 1/120
Processed 2/120
Processed 3/120
Processed 4/120
Processed 5/120
Processed 6/120
Processed 7/120
Processed 8/120
Processed 9/120
Processed 10/120
Processed 11/120
Processed 12/120
Processed 13/120
Processed 14/120
Processed 15/120
Processed 16/120
Processed 17/120
Processed 18/120
Processed 19/120
Processed 20/120
Processed 21/120
Processed 22/120
Processed 23/120
Processed 24/120
Processed 25/120
Processed 26/120
Processed 27/120
Processed 28/120
Processed 29/120
Processed 30/120
Processed 31/120
Processed 32/120
Processed 33/120
Processed 34/120
Processed 35/120
Processed 36/120
Processed 37/120
Processed 38/120
Processed 39/120
Processed 40/120
Processed 41/120
Processed 42/120
Processed 43/120
Processed 44/120
Processed 45/120
Processed 46/120
Processed 47/120
Processed 48/120
Processed 49/120
Processed 50/120
Processed 51/120
Processed 52/120
Processed 53/120
Processed 54/120
Processed 55/120
Processed 56/120
Processed 57/120
Processed 58/120
Processed 59/120
Proces

In [33]:
rag_test_df = pd.DataFrame(rag_test_results)

rag_test_df.head()

,numero_aviso,actual,prediction
0,92334,OBJETADO,OBJETADO
1,206854,OBJETADO,ENTREGADO
2,237284,OBJETADO,OBJETADO
3,231175,ENTREGADO,ENTREGADO
4,79950,OBJETADO,OBJETADO


In [34]:
rag_test_cm = get_confusion_matrix(
    rag_test_df["actual"],
    rag_test_df["prediction"]
)

rag_test_metrics = evaluate_predictions(
    rag_test_df["actual"],
    rag_test_df["prediction"]
)

tn, fp, fn, tp = rag_test_cm.ravel()

rag_test_specificity = tn / (tn + fp)

print("RAG test confusion matrix:")
print(rag_test_cm)

print("\nRAG test metrics:")
print(rag_test_metrics)

print("\nSpecificity:", rag_test_specificity)

RAG test confusion matrix:
[[23 22]
 [11 64]]

RAG test metrics:
{'precision': 0.7441860465116279, 'recall': 0.8533333333333334, 'f1': 0.7950310559006211, 'f2': 0.8290155440414507}

Specificity: 0.5111111111111111


In [35]:
RAG_TEST_RESULTS_PATH = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "rag_test_results.csv"
)

rag_test_df.to_csv(
    RAG_TEST_RESULTS_PATH,
    index=False
)

print("Saved to:")
print(RAG_TEST_RESULTS_PATH)

Saved to:
/home/sagemaker-user/subocol-ia/data/processed/rag_test_results.csv


## Final Model Comparison — Baseline vs RAG

In [36]:
baseline_tn, baseline_fp, baseline_fn, baseline_tp = test_cm.ravel()

baseline_specificity = (
    baseline_tn
    / (baseline_tn + baseline_fp)
)

In [37]:
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Bedrock RAG"
    ],
    "Precision": [
        test_metrics["precision"],
        rag_test_metrics["precision"]
    ],
    "Recall": [
        test_metrics["recall"],
        rag_test_metrics["recall"]
    ],
    "F1": [
        test_metrics["f1"],
        rag_test_metrics["f1"]
    ],
    "F2": [
        test_metrics["f2"],
        rag_test_metrics["f2"]
    ],
    "Specificity": [
        baseline_specificity,
        rag_test_specificity
    ]
})

comparison_df.round(4)

,Model,Precision,Recall,F1,F2,Specificity
0,Logistic Regression,0.6228,0.9467,0.7513,0.8575,0.0444
1,Bedrock RAG,0.7442,0.8533,0.7950,0.8290,0.5111


In [38]:
COMPARISON_PATH = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "model_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)

print("Saved to:")
print(COMPARISON_PATH)

Saved to:
/home/sagemaker-user/subocol-ia/data/processed/model_comparison.csv


In [39]:
from src.api.schemas import ClaimRequest
from src.api.inference import prepare_claim_for_inference

In [40]:
test_claim = test_df.iloc[0]

In [41]:
claim_request = ClaimRequest(
    version_hechos=test_claim["version_hechos"],
    piezas=test_claim["parts"],
    marca=test_claim["marca"],
    linea=test_claim["linea"],
    version=test_claim["version"],
    modelo=int(test_claim["modelo"]),
    fecha_creacion=test_claim["fecha_creacion"]
)

In [42]:
inference_df = prepare_claim_for_inference(
    claim_request
)

inference_df

,version_hechos,parts_text,marca,linea,version,modelo,fecha_creacion,vehicle_age,piezas_totales,valid_parts
0,reparacion de parte trasera y lado derecho.,puerta trasero derecho guardafango trasero der...,TOYOTA,PRADO,[1],2023,2023-10-24 14:16:53.951,0,3,3


In [43]:
print("Original pieces:")
print(test_claim["parts"])

print("\nGenerated parts_text:")
print(inference_df.iloc[0]["parts_text"])

print("\nVehicle age:")
print(inference_df.iloc[0]["vehicle_age"])

print("\nTotal pieces:")
print(inference_df.iloc[0]["piezas_totales"])

Original pieces:
['puerta trasero derecho', 'guardafango trasero derecho', 'defensa trasero']

Generated parts_text:
puerta trasero derecho guardafango trasero derecho defensa trasero

Vehicle age:
0

Total pieces:
3


In [44]:
from src.api.inference import (
    prepare_claim_for_inference,
    predict_with_baseline
)

In [45]:
baseline_inference = predict_with_baseline(
    inference_df,
    MODEL_PATH
)

baseline_inference

{'prediction': 'OBJETADO', 'probability_objetado': 0.4596203166691088}

In [46]:
print(
    "Prediction:",
    baseline_inference["prediction"]
)

print(
    "Probability OBJETADO:",
    baseline_inference["probability_objetado"]
)

print(
    "Actual label:",
    test_claim["estado_aviso"]
)

Prediction: OBJETADO
Probability OBJETADO: 0.4596203166691088
Actual label: OBJETADO


In [47]:
from src.api.inference import (
    prepare_claim_for_inference,
    predict_with_baseline,
    predict_with_rag
)

In [48]:
rag_inference = predict_with_rag(
    claim_df=inference_df,
    knowledge_base_id=BEDROCK_KB_ID,
    number_of_results=5
)

rag_inference

{'prediction': 'OBJETADO', 'retrieved_claims': 5}

In [49]:
print("Baseline prediction:")
print(baseline_inference)

print("\nRAG prediction:")
print(rag_inference)

print("\nActual label:")
print(test_claim["estado_aviso"])

Baseline prediction:
{'prediction': 'OBJETADO', 'probability_objetado': 0.4596203166691088}

RAG prediction:
{'prediction': 'OBJETADO', 'retrieved_claims': 5}

Actual label:
OBJETADO


In [50]:
import json

from src.api.lambda_handler import lambda_handler

In [51]:
api_body = {
    "version_hechos": test_claim["version_hechos"],
    "piezas": test_claim["parts"],
    "marca": test_claim["marca"],
    "linea": test_claim["linea"],
    "version": test_claim["version"],
    "modelo": int(test_claim["modelo"]),
    "fecha_creacion": test_claim["fecha_creacion"].isoformat()
}

In [52]:
baseline_event = {
    "rawPath": "/predict/baseline",
    "body": json.dumps(api_body)
}

In [53]:
baseline_lambda_response = lambda_handler(
    baseline_event,
    None
)

baseline_lambda_response

{'statusCode': 200,
 'headers': {'Content-Type': 'application/json'},
 'body': '{"model": "baseline", "prediction": "OBJETADO", "probability_objetado": 0.4596203166691088}'}

In [54]:
baseline_response_body = json.loads(
    baseline_lambda_response["body"]
)

baseline_response_body

{'model': 'baseline',
 'prediction': 'OBJETADO',
 'probability_objetado': 0.4596203166691088}

In [59]:
print(
    "Lambda prediction:",
    baseline_response_body["prediction"]
)

print(
    "Actual label:",
    test_claim["estado_aviso"]
)

Lambda prediction: OBJETADO
Actual label: OBJETADO


In [60]:
compare_event = {
    "rawPath": "/predict/compare",
    "body": json.dumps(api_body)
}

In [61]:
compare_lambda_response = lambda_handler(
    compare_event,
    None
)

In [62]:
compare_response_body = json.loads(
    compare_lambda_response["body"]
)

compare_response_body

{'baseline': {'prediction': 'OBJETADO',
  'probability_objetado': 0.4596203166691088},
 'rag': {'prediction': 'OBJETADO', 'retrieved_claims': 5}}